## 1. Modèles de Markov Cachés (HMM) et Algorithme de Viterbi

L'objectif est de trouver la séquence d'étiquettes $Y$ qui maximise la probabilité étant donné les observations $X$:
$$ \hat{Y} = \argmax_{Y} P(X|Y)P(Y) $$

L'algorithme de Viterbi évite l'explosion combinatoire en calculant récursivement la probabilité du meilleur chemin partiel menant à l'état $q_j$ à l'instant $t$[cite: 268, 269]:
$$ v_t(j) = \max_{i=1}^{N} v_{t-1}(i) a_{ij} b_j(o_t) $$

* $v_{t-1}(i)$ : probabilité du chemin précédent[cite: 270].
* $a_{ij}$ : probabilité de transition de l'état $q_i$ vers $q_j$[cite: 270].
* $b_j(o_t)$ : probabilité d'émission de l'observation $o_t$ par l'état $q_j$.

In [5]:
import numpy as np

# Exemple simplifié basé sur : "Janet will back the bill"
observations = ["Janet", "will", "back"]
etats = ["NNP", "MD", "VB"]

# Probabilités initiales (pi)
start_p = {"NNP": 0.28, "MD": 0.0006, "VB": 0.0031}

# Matrice de transition a_ij (ligne: état t-1, colonne: état t)
trans_p = {
    "NNP": {"NNP": 0.3777, "MD": 0.0110, "VB": 0.0009},
    "MD":  {"NNP": 0.0008, "MD": 0.0002, "VB": 0.7968},
    "VB":  {"NNP": 0.0322, "MD": 0.0005, "VB": 0.0050}
}

# Matrice d'émission b_j(o_t)
emit_p = {
    "NNP": {"Janet": 0.000032, "will": 0.0, "back": 0.000048},
    "MD":  {"Janet": 0.0, "will": 0.308431, "back": 0.0},
    "VB":  {"Janet": 0.0, "will": 0.000028, "back": 0.000672}
}

def viterbi(obs, states, start_p, trans_p, emit_p):
    V = [{}]
    path = {}

    # Initialisation t=0
    for y in states:
        V[0][y] = start_p[y] * emit_p[y].get(obs[0], 0)
        path[y] = [y]

    # Récursion t > 0
    for t in range(1, len(obs)):
        V.append({})
        newpath = {}

        for y in states:
            # Recherche du max
            (prob, state) = max(
                (V[t-1][y0] * trans_p[y0][y] * emit_p[y].get(obs[t], 0), y0)
                for y0 in states
            )
            V[t][y] = prob
            newpath[y] = path[state] + [y]

        path = newpath

    # Terminaison
    n = len(obs) - 1
    (prob, state) = max((V[n][y], y) for y in states)
    return prob, path[state]

probabilite, sequence = viterbi(observations, etats, start_p, trans_p, emit_p)
print(f"Séquence optimale : {sequence}")
print(f"Probabilité du chemin : {probabilite}")

Séquence optimale : ['NNP', 'MD', 'VB']
Probabilité du chemin : 1.6277110629728254e-11


## 2. Champs Aléatoires Conditionnels (CRF)

Contrairement aux HMM, les CRF sont discriminatifs et s'affranchissent de l'indépendance des observations. [cite_start]Pour une séquence $X$ et une séquence d'étiquettes $Y$, la probabilité est définie par[cite: 486, 487]:

$$ P(Y|X) = \frac{1}{Z(X)} \exp\left( \sum_{i,k} \lambda_k f_k(y_{i-1}, y_i, X, i) \right) $$

* $f_k$ : fonctions de caractéristiques évaluées sur le contexte global[cite: 488].
* $\\lambda_k$ : poids appris lors de l'entraînement[cite: 489].
* $Z(X)$ : fonction de partition servant de facteur de normalisation[cite: 490].

In [6]:
def word2features(sent, i):
    word = sent[i]
    
    # Implémentation des fonctions fk du cours
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word.isupper()': word.isupper(), # ex: isCapitalized(w)
        'word.istitle()': word.istitle(),
        'word[-3:]': word[-3:],           # Suffixes
        'word[:3]': word[:3],             # Préfixes
    }
    
    # Intégration du contexte (mot précédent)
    if i > 0:
        word1 = sent[i-1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
        })
    else:
        features['BOS'] = True # Beginning of Sequence

    return features

phrase = ["Mark", "Watney", "visits", "Mars"]
print("Caractéristiques pour le mot 'Watney' :")
print(word2features(phrase, 1))

Caractéristiques pour le mot 'Watney' :
{'bias': 1.0, 'word.lower()': 'watney', 'word.isupper()': False, 'word.istitle()': True, 'word[-3:]': 'ney', 'word[:3]': 'Wat', '-1:word.lower()': 'mark', '-1:word.istitle()': True}


## 3. GLiNER : Modèle NER Généraliste

GLiNER s'appuie sur un modèle de langage bidirectionnel pour modéliser simultanément les textes et les types d'entités (définis librement) dans un espace vectoriel[cite: 717, 720, 722].

La correspondance entre un type d'entité et un segment de texte repose sur une fonction d'activation sigmoïde appliquée à leur produit scalaire[cite: 782, 783, 784]:

$$ \phi(i,j,t) = \sigma(S_{ij}^T q_t) \in \mathbb{R} $$

* $S_{ij}$ : représentation matricielle du segment allant de $i$ à $j$[cite: 774].
* $q_t$ : embedding du type d'entité $t$[cite: 785].

In [4]:
from gliner import GLiNER
    
# Chargement d'un petit modèle GLiNER
model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")

text = "Alain Farley works at McGill University"

# Les catégories sont libres et non figées, contrairement aux anciens CRF
labels = ["Person", "Organization", "Location"]

entities = model.predict_entities(text, labels)

for entity in entities:
    print(f"Entité : {entity['text']} | Type : {entity['label']} | Score : {entity['score']:.3f}")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Entité : Alain Farley | Type : Person | Score : 0.986
Entité : McGill University | Type : Organization | Score : 0.831


In [7]:
import numpy as np

# Simulation avec 3 états (0: NNP, 1: MD, 2: VB) et 3 observations
N_etats = 3
T_obs = 3

# Probabilités (fictives, transformées en log pour la stabilité)
# Remplacer les 0 par une valeur epsilon (1e-10) avant le log pour éviter log(0)
eps = 1e-10

pi = np.log(np.array([0.28, 0.0006, 0.0031]) + eps)

# Matrice de transition A (N x N)
A = np.log(np.array([
    [0.3777, 0.0110, 0.0009],
    [0.0008, 0.0002, 0.7968],
    [0.0322, 0.0005, 0.0050]
]) + eps)

# Matrice d'émission B (N x T_obs) 
# Lignes = états, Colonnes = séquence d'observations observée t=0, t=1, t=2
B = np.log(np.array([
    [0.000032, eps, 0.000048],
    [eps, 0.308431, eps],
    [eps, 0.000028, 0.000672]
]))

def viterbi_log(pi, A, B):
    N, T = B.shape
    V = np.zeros((N, T))
    backpointer = np.zeros((N, T), dtype=int)
    
    # [cite_start]Initialisation [cite: 251]
    V[:, 0] = pi + B[:, 0]
    
    # [cite_start]Récursion [cite: 256]
    for t in range(1, T):
        for j in range(N):
            # Calcul des probabilités pour tous les états précédents i vers j
            trans_probs = V[:, t-1] + A[:, j] + B[j, t]
            V[j, t] = np.max(trans_probs)
            backpointer[j, t] = np.argmax(trans_probs)
            
    # [cite_start]Terminaison [cite: 259, 262]
    best_path_prob = np.max(V[:, T-1])
    best_last_state = np.argmax(V[:, T-1])
    
    # Backtracking
    path = [best_last_state]
    for t in range(T-1, 0, -1):
        path.insert(0, backpointer[path[0], t])
        
    return best_path_prob, path

prob, sequence = viterbi_log(pi, A, B)
print(f"Séquence d'états optimale (indices) : {sequence}")
print(f"Log-probabilité du chemin : {prob:.4f}")

Séquence d'états optimale (indices) : [0, 1, 2]
Log-probabilité du chemin : -24.8413


In [8]:
def evaluate_sequence(y_true, y_pred, target_class):
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == target_class and yp == target_class)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != target_class and yp == target_class)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == target_class and yp != target_class)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0  # [cite: 350]
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0     # [cite: 351]
    
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall) # [cite: 352]
    else:
        f1 = 0.0
        
    return precision, recall, f1

gold = ["B-PER", "I-PER", "O", "B-ORG", "I-ORG"]
pred = ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"]

p, r, f1 = evaluate_sequence(gold, pred, "B-PER")
print(f"Métriques pour B-PER -> Précision: {p:.2f}, Rappel: {r:.2f}, F1: {f1:.2f}")

p, r, f1 = evaluate_sequence(gold, pred, "B-ORG")
print(f"Métriques pour B-ORG -> Précision: {p:.2f}, Rappel: {r:.2f}, F1: {f1:.2f}")

Métriques pour B-PER -> Précision: 1.00, Rappel: 1.00, F1: 1.00
Métriques pour B-ORG -> Précision: 0.00, Rappel: 0.00, F1: 0.00


In [9]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Dimensions arbitraires de l'espace latent
d_model = 64 

# [cite_start]1. Embeddings simulés depuis le transformateur bidirectionnel [cite: 741]
h_i = np.random.randn(d_model) # Embedding du premier mot du span
h_j = np.random.randn(d_model) # Embedding du dernier mot du span

# [cite_start]2. Simulation du FFN pour obtenir S_ij [cite: 775]
# L'opérateur de concaténation/fusion (otimes) est simplifié ici par une concaténation
h_concat = np.concatenate([h_i, h_j])
W_ffn = np.random.randn(d_model * 2, d_model) / np.sqrt(d_model)
S_ij = np.dot(h_concat, W_ffn) # S_ij de dimension (d_model,)

# [cite_start]3. Embeddings des types d'entités (q_t) [cite: 785]
q_person = np.random.randn(d_model)
q_org = np.random.randn(d_model)

# [cite_start]4. Calcul de la probabilité de correspondance phi(i,j,t) [cite: 784]
score_person = np.dot(S_ij.T, q_person)
score_org = np.dot(S_ij.T, q_org)

prob_person = sigmoid(score_person)
prob_org = sigmoid(score_org)

print(f"Probabilité que le segment [i,j] soit 'Person' : {prob_person:.4f}")
print(f"Probabilité que le segment [i,j] soit 'Organization' : {prob_org:.4f}")

# [cite_start]Décodage (Flat-NER) : on sélectionnerait le type avec le score max dépassant un seuil [cite: 799]
seuil = 0.5
if prob_person > seuil or prob_org > seuil:
    prediction = 'Person' if prob_person > prob_org else 'Organization'
    print(f"-> Classification finale pour ce span : {prediction}")
else:
    print("-> Aucune entité détectée (O)")

Probabilité que le segment [i,j] soit 'Person' : 0.0103
Probabilité que le segment [i,j] soit 'Organization' : 1.0000
-> Classification finale pour ce span : Organization
